In [ ]:
import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

from config import RunConfig
from utils.ptp_utils import AttentionStore
from utils import ptp_utils

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using device: {device}")

SD_VERSION = "CompVis/stable-diffusion-v1-4"  # or "stabilityai/stable-diffusion-2-1-base"

# Standard SD — no guidance
from pipeline import StormPipeline
sd_pipeline = StormPipeline.from_pretrained(SD_VERSION).to(device)

# Poisson pipeline
from poisson_pipeline import PoissonPipeline
poisson_pipeline = PoissonPipeline.from_pretrained(SD_VERSION).to(device)

In [ ]:
config = RunConfig(
    n_inference_steps = 50,
    guidance_scale    = 7.5,
    max_iter_to_alter = 25,
    attention_res     = 16,
    smooth_attentions = True,
    sigma             = 0.5,
    kernel_size       = 3,
    scale_factor      = 20,
    scale_range       = (1.0, 0.5),
    thresholds        = {0: 0.9, 5: 0.95, 10: 0.99, 15: 0.995, 20: 0.999},
)

In [ ]:
from run import get_noun_indices_to_alter

prompt = "a cake to the left of a suitcase"
token_indices = get_noun_indices_to_alter(poisson_pipeline, prompt)
print(f"Token indices: {token_indices}")


In [ ]:
def run_generation(pipeline, prompt, token_indices, seed, config, run_standard=False):
    """Run one image generation and return the PIL image."""
    g = torch.Generator("cuda").manual_seed(seed)
    controller = AttentionStore()
    ptp_utils.register_attention_control(pipeline, controller)

    outputs = pipeline(
        prompt              = prompt,
        attention_store     = controller,
        indices_to_alter    = token_indices,
        attention_res       = config.attention_res,
        guidance_scale      = config.guidance_scale,
        generator           = g,
        num_inference_steps = config.n_inference_steps,
        max_iter_to_alter   = config.max_iter_to_alter,
        run_standard_sd     = run_standard,
        thresholds          = config.thresholds,
        scale_factor        = config.scale_factor,
        scale_range         = config.scale_range,
        smooth_attentions   = config.smooth_attentions,
        sigma               = config.sigma,
        kernel_size         = config.kernel_size,
        sd_2_1              = False,
    )
    return outputs.images[0]


In [ ]:
SEED = 42

print("Running standard SD...")
img_sd = run_generation(
    sd_pipeline, prompt, token_indices, SEED, config, run_standard=True
)

print("Running Poisson pipeline...")
img_poisson = run_generation(
    poisson_pipeline, prompt, token_indices, SEED, config, run_standard=False
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(img_sd)
axes[0].set_title("Standard SD", fontsize=14)
axes[0].axis("off")

axes[1].imshow(img_poisson)
axes[1].set_title("Poisson (Ours)", fontsize=14)
axes[1].axis("off")

fig.suptitle(f'"{prompt}"', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("poisson_vs_sd.png", bbox_inches="tight")
plt.show()

In [ ]:
SEEDS = [0, 1, 2, 3]

results_sd      = []
results_poisson = []

for seed in SEEDS:
    print(f"Seed {seed}...")
    results_sd.append(
        run_generation(sd_pipeline, prompt, token_indices, seed, config, run_standard=True)
    )
    results_poisson.append(
        run_generation(poisson_pipeline, prompt, token_indices, seed, config, run_standard=False)
    )

fig, axes = plt.subplots(2, len(SEEDS), figsize=(16, 8))

for i, (img_sd, img_p) in enumerate(zip(results_sd, results_poisson)):
    axes[0, i].imshow(img_sd)
    axes[0, i].set_title(f"SD seed={SEEDS[i]}", fontsize=10)
    axes[0, i].axis("off")

    axes[1, i].imshow(img_p)
    axes[1, i].set_title(f"Poisson seed={SEEDS[i]}", fontsize=10)
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Standard SD", fontsize=12)
axes[1, 0].set_ylabel("Poisson", fontsize=12)

fig.suptitle(f'"{prompt}"', fontsize=13)
plt.tight_layout()
plt.savefig("poisson_vs_sd_seeds.png", bbox_inches="tight")
plt.show()

In [ ]:
from utils.ptp_utils import aggregate_attention

g = torch.Generator("cuda").manual_seed(SEED)
controller = AttentionStore()
ptp_utils.register_attention_control(poisson_pipeline, controller)

img_debug = run_generation(
    poisson_pipeline, prompt, token_indices, SEED, config, run_standard=False
)

# Aggregate attention after generation
attn_maps = aggregate_attention(
    attention_store = controller,
    res             = 16,
    from_where      = ("up", "down", "mid"),
    is_cross        = True,
    select          = 0,
)
print(f"Attention maps shape: {attn_maps.shape}")
# Expected: (16, 16, num_tokens) — verify this matches what _poisson_loss expects

# Visualize attention for each noun token
noun_indices = token_indices[0]  # subject indices
fig, axes = plt.subplots(1, len(noun_indices), figsize=(6 * len(noun_indices), 5))

if len(noun_indices) == 1:
    axes = [axes]

tokens = poisson_pipeline.tokenizer(prompt)['input_ids']
token_words = [poisson_pipeline.tokenizer.decode(t) for t in tokens]

for ax, idx in zip(axes, noun_indices):
    attn = attn_maps[:, :, idx - 1].detach().cpu().numpy()  # shift by 1
    im = ax.imshow(attn, cmap="hot")
    ax.set_title(f"Token: '{token_words[idx]}'", fontsize=12)
    ax.axis("off")
    plt.colorbar(im, ax=ax)

plt.suptitle("Cross-attention maps after Poisson generation", fontsize=13)
plt.tight_layout()
plt.show()